# Tema 4 (Laboratorul 5) — Modele Markov ascunse (Algoritmii Forward și Viterbi)

---
## Exercițiul 1 — Algoritmul Forward (Demonstrație teoretică)

Considerăm un Model Markov Ascuns (HMM) cu:
* stările ascunse $X_1, \dots, X_T$ și observațiile $O_1, \dots, O_T$;
* distribuția inițială $\pi_i = P(X_1 = i)$;
* probabilitățile de tranziție $a_{ij} = P(X_{t+1} = j \mid X_t = i)$;
* probabilitățile de emisie $b_j(o) = P(O_t = o \mid X_t = j)$.

Variabila Forward este definită prin:
$$
\alpha_t(j) = P(O_1 = o_1, \dots, O_t = o_t, X_t = j).
$$

### a) Inițializare ($t = 1$)
Pentru $t = 1$, aplicăm definiția probabilității condiționate $P(A \cap B) = P(B)P(A \mid B)$:
$$
\alpha_1(j) = P(O_1 = o_1, X_1 = j) = P(X_1 = j) \cdot P(O_1 = o_1 \mid X_1 = j) = \pi_j b_j(o_1).
$$

### b) Recurență ($t = 2, \dots, T$)
Aplicăm **formula probabilității totale** condiționând după toate valorile posibile $i$ ale stării ascunse anterioare $X_{t-1}$:
$$
\begin{aligned}
\alpha_t(j) &= P(O_1 = o_1, \dots, O_t = o_t, X_t = j) \\
&= \sum_{i} P(O_1 = o_1, \dots, O_{t-1} = o_{t-1}, X_{t-1} = i, X_t = j, O_t = o_t).
\end{aligned}
$$
Folosind regula produsului (înlănțuirea probabilităților condiționate), fiecare termen al sumei se scrie:
$$
\begin{aligned}
P(O_{1:t-1} = o_{1:t-1}, X_{t-1} = i, X_t = j, O_t = o_t)
&= P(O_{1:t-1} = o_{1:t-1}, X_{t-1} = i) \\
&\quad \cdot P(X_t = j \mid O_{1:t-1} = o_{1:t-1}, X_{t-1} = i) \\
&\quad \cdot P(O_t = o_t \mid O_{1:t-1} = o_{1:t-1}, X_{t-1} = i, X_t = j).
\end{aligned}
$$
Aplicăm cele două ipoteze fundamentale ale unui HMM:
1. **Proprietatea Markov de ordinul 1** pentru stările ascunse: $P(X_t = j \mid O_{1:t-1}, X_{t-1} = i) = P(X_t = j \mid X_{t-1} = i) = a_{ij}$.
2. **Independența condiționată a observației curente** față de trecut, dată fiind starea curentă $X_t$: $P(O_t = o_t \mid O_{1:t-1}, X_{t-1}, X_t = j) = P(O_t = o_t \mid X_t = j) = b_j(o_t)$.

Substituind în sumă și scoțând factorul comun $b_j(o_t)$ în fața sumei după $i$:
$$
\alpha_t(j) = \sum_{i} \alpha_{t-1}(i) \cdot a_{ij} \cdot b_j(o_t) = b_j(o_t) \sum_{i} \alpha_{t-1}(i) a_{ij}, \quad t = 2, \dots, T. \qquad \blacksquare
$$

### c) Finalizare
Deoarece evenimentele $\{X_T = j\}$ (pentru toate stările $j$) sunt disjuncte două câte două și formează o partiție a spațiului de probă, aplicând **legea probabilității totale** obținem:
$$
P(O_1 = o_1, \dots, O_T = o_T) = \sum_{j} P(O_1 = o_1, \dots, O_T = o_T, X_T = j) = \sum_{j} \alpha_T(j). \qquad \blacksquare
$$

---
## Exercițiul 2 — Implementarea de la zero a algoritmului Viterbi

### a) Definirea modelului
* **Stările ascunse** (în ordinea `D, M, U`):
  $$\pi = \left[\frac{1}{3}, \frac{1}{3}, \frac{1}{3}\right].$$
* **Matricea de tranziție $A$** (rânduri și coloane: `D, M, U`):
  $$A = \begin{pmatrix} 0 & 0.5 & 0.5 \\ 0.5 & 0.25 & 0.25 \\ 0.5 & 0.25 & 0.25 \end{pmatrix}.$$
* **Matricea de emisie $B$** (rânduri: `D, M, U`; coloane: `FB, B, S, NS`):
  $$B = \begin{pmatrix} 0.10 & 0.20 & 0.40 & 0.30 \\ 0.15 & 0.25 & 0.50 & 0.10 \\ 0.20 & 0.30 & 0.40 & 0.10 \end{pmatrix}.$$

### c) Rolul matricei $\psi_t(j)$
Matricea de **backpointere** $\psi_t(j) = \arg\max_i \big(\delta_{t-1}(i) a_{ij}\big)$ memorează, pentru fiecare pas de timp $t \ge 2$ și fiecare stare curentă $j$, indicele stării anterioare $i^*$ din care se ajunge în $j$ cu probabilitate maximă. La finalul parcurgerii directe ($t = T$), $\psi$ permite **reconstrucția în ordine inversă (backtracking)** a secvenței optime de stări $x_T^*, x_{T-1}^*, \dots, x_1^*$.

### d) Verificarea manuală a valorilor $\delta_1$ și $\delta_2$ (primele două note: $o_1 = \text{FB}, \; o_2 = \text{FB}$)
* **Pasul $t = 1$ ($o_1 = \text{FB}$, coloana $0$ din $B$):**
  $$\delta_1(D) = \pi_D b_D(\text{FB}) = \frac{1}{3} \times 0.10 = \frac{0.10}{3} \approx 0.033333,$$
  $$\delta_1(M) = \pi_M b_M(\text{FB}) = \frac{1}{3} \times 0.15 = \frac{0.15}{3} = 0.050000,$$
  $$\delta_1(U) = \pi_U b_U(\text{FB}) = \frac{1}{3} \times 0.20 = \frac{0.20}{3} \approx 0.066667.$$

* **Pasul $t = 2$ ($o_2 = \text{FB}$):**
  * **Pentru starea $D$ ($b_D(\text{FB}) = 0.10$):**
    $$\max_{i \in \{D, M, U\}} \big(\delta_1(i) a_{iD}\big) = \max\left(\frac{0.10}{3}\cdot 0, \; 0.05 \cdot 0.5, \; \frac{0.20}{3}\cdot 0.5\right) = \max\left(0, \; 0.025, \; \frac{0.10}{3}\right) = \frac{0.10}{3} \approx 0.033333 \quad (\text{atins pentru } i^* = U).$$
    $$\delta_2(D) = 0.10 \times \frac{0.10}{3} = \frac{0.01}{3} \approx 0.003333.$$
  * **Pentru starea $M$ ($b_M(\text{FB}) = 0.15$):**
    $$\max_{i} \big(\delta_1(i) a_{iM}\big) = \max\left(\frac{0.10}{3}\cdot 0.5, \; 0.05 \cdot 0.25, \; \frac{0.20}{3}\cdot 0.25\right) = \max\left(\frac{0.05}{3}, \; 0.0125, \; \frac{0.05}{3}\right) = \frac{0.05}{3} \approx 0.016667 \quad (i^* \in \{D, U\}).$$
    $$\delta_2(M) = 0.15 \times \frac{0.05}{3} = 0.002500.$$
  * **Pentru starea $U$ ($b_U(\text{FB}) = 0.20$):**
    $$\max_{i} \big(\delta_1(i) a_{iU}\big) = \max\left(\frac{0.10}{3}\cdot 0.5, \; 0.05 \cdot 0.25, \; \frac{0.20}{3}\cdot 0.25\right) = \frac{0.05}{3} \approx 0.016667 \quad (i^* \in \{D, U\}).$$
    $$\delta_2(U) = 0.20 \times \frac{0.05}{3} = \frac{0.01}{3} \approx 0.003333.$$

In [ ]:
import numpy as np

# a) Definirea modelului HMM
stari = ["D", "M", "U"]
observatii = ["FB", "B", "S", "NS"]

pi = np.array([1/3, 1/3, 1/3])
A = np.array([
    [0.0, 0.5,  0.5],
    [0.5, 0.25, 0.25],
    [0.5, 0.25, 0.25]
])
B = np.array([
    [0.10, 0.20, 0.40, 0.30],
    [0.15, 0.25, 0.50, 0.10],
    [0.20, 0.30, 0.40, 0.10]
])

# Verificarea normalizării rândurilor
assert np.isclose(np.sum(pi), 1.0), "Suma distribuției inițiale pi trebuie să fie 1!"
assert np.allclose(np.sum(A, axis=1), 1.0), "Fiecare rând din A trebuie să aibă suma 1!"
assert np.allclose(np.sum(B, axis=1), 1.0), "Fiecare rând din B trebuie să aibă suma 1!"
print("a) Verificare normalizare: sum(pi) = 1, sumele pe rânduri în A = [1, 1, 1], în B = [1, 1, 1].\n")

# Secvența observată
secventa_obs = ["FB", "FB", "S", "B", "B", "S", "B", "B", "NS", "B", "B", "S"]
obs_idx = [observatii.index(o) for o in secventa_obs]

def viterbi_manual(pi, A, B, obs_idx):
    T = len(obs_idx)
    N = len(pi)
    delta = np.zeros((T, N))
    psi = np.zeros((T, N), dtype=int)

    # b) Inițializare (t = 0 în indexare 0-based, corespunzător t = 1)
    delta[0] = pi * B[:, obs_idx[0]]

    # c) Recurență (t = 1 .. T-1)
    for t in range(1, T):
        for j in range(N):
            tranzitii = delta[t - 1] * A[:, j]
            psi[t, j] = int(np.argmax(tranzitii))
            delta[t, j] = B[j, obs_idx[t]] * np.max(tranzitii)

    # e) Reconstrucția traseului optim (backtracking)
    traseu_idx = np.zeros(T, dtype=int)
    traseu_idx[-1] = int(np.argmax(delta[-1]))
    prob_comuna_max = float(delta[-1, traseu_idx[-1]])

    for t in range(T - 1, 0, -1):
        traseu_idx[t - 1] = psi[t, traseu_idx[t]]

    return delta, psi, traseu_idx, prob_comuna_max

delta, psi, traseu_idx, prob_max = viterbi_manual(pi, A, B, obs_idx)
secventa_optima = [stari[i] for i in traseu_idx]

print("d) Valori intermediare:")
print(f"   delta_1 (D, M, U) = {np.round(delta[0], 6)}")
print(f"   delta_2 (D, M, U) = {np.round(delta[1], 6)}")
print(f"   psi_2   (D, M, U) = {[stari[i] for i in psi[1]]}\n")

print("e) Rezultatul final Viterbi:")
print(f"   Secvența observată:       {secventa_obs}")
print(f"   Secvența optimă de stări: {secventa_optima}")
print(f"   Probabilitatea comună maximă max P(x1..x12, o1..o12) = {prob_max:.10e}")

# Opțional: Algoritmul Forward pentru P(o) și probabilitatea condiționată P(x* | o)
def forward_manual(pi, A, B, obs_idx):
    T = len(obs_idx)
    N = len(pi)
    alpha = np.zeros((T, N))
    alpha[0] = pi * B[:, obs_idx[0]]
    for t in range(1, T):
        for j in range(N):
            alpha[t, j] = B[j, obs_idx[t]] * np.sum(alpha[t - 1] * A[:, j])
    return alpha, float(np.sum(alpha[-1]))

_, prob_obs = forward_manual(pi, A, B, obs_idx)
prob_cond = prob_max / prob_obs
print(f"\nOpțional (Forward + Posterior):")
print(f"   P(o_1..o_12)       = {prob_obs:.10e}")
print(f"   P(x* | o_1..o_12)  = {prob_cond:.6f} ({prob_cond*100:.4f}%)")

### f) Interpretare
* **De ce Viterbi caută cea mai probabilă secvență completă, nu dificultatea optimă separat la fiecare test?**
  Decodarea punctuală (*posterior decoding*, $\hat{x}_t = \arg\max_j P(X_t = j \mid O)$) optimizează fiecare pas individual fără a ține cont de probabilitățile de tranziție dintre stări consecutive. Într-un model cu tranziții interzise (aici $a_{DD} = P(X_{t+1} = D \mid X_t = D) = 0$, deoarece profesorul nu dă niciodată două teste dificile la rând), decodarea individuală ar putea genera o secvență imposibilă cu probabilitate globală $0$. Algoritmul **Viterbi** maximizează probabilitatea întregii traiectorii $(x_1, \dots, x_{12})$, garantând consistența tranzițiilor pe tot parcursul secvenței.
* **Natura scorului obținut la punctul e):**
  Valoarea obținută $\delta_{12}(x_{12}^*) = \max_{x_1, \dots, x_{12}} P(x_1, \dots, x_{12}, o_1, \dots, o_{12}) \approx 2.109375 \times 10^{-11}$ reprezintă **probabilitatea comună** a traseului optim împreună cu secvența de observații $P(x^*, o)$, **nu** probabilitatea condiționată $P(x^* \mid o)$. Pentru a obține probabilitatea condiționată a traseului dat fiind șirul notelor, se împarte scorul Viterbi la probabilitatea totală a observațiilor $P(o)$ calculată prin algoritmul Forward:
  $$P(x^* \mid o) = \frac{P(x^*, o)}{P(o)} \approx \frac{2.109375 \times 10^{-11}}{7.325943 \times 10^{-8}} \approx 0.000288 \quad (0.0288\%).$$